# Bakehouse Customers

**Dataset:** `samples.bakehouse.sales_customers`

**Difficulty:** Easy

**Topics:** groupBy, filter, distinct, null handling

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — groupBy, filter, distinct

| Function | What it does |
|----------|-------------|
| `df.groupBy(col).count()` | Counts rows per group — the most common aggregation |
| `df.groupBy(col).agg(F.count("*"))` | Equivalent to `.count()` but lets you add more aggregations |
| `df.filter(F.col(c).isNull())` | Keeps only rows where the column is null |
| `df.filter(F.col(c).isNotNull())` | Keeps only rows where the column has a value |
| `df.distinct()` | Removes duplicate rows |
| `df.select(col).distinct().count()` | Counts unique values in a column |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) · [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_customers")

# Count rows per city (not per country — the problems ask about country)
df.groupBy("city").agg(
    F.count("*").alias("num_customers")
).orderBy(F.col("num_customers").desc()).show(5)

# How many unique email domains exist?
print("Unique emails:", df.select("email_address").distinct().count())

# Check for nulls in a specific column
null_count = df.filter(F.col("city").isNull()).count()
print("Rows with null city:", null_count)

## Problem 1

Count the number of customers in each **country**, sorted from most to fewest.
Load `samples.bakehouse.sales_customers` and group by `country`.

**Expected output columns:**
- `country` - country name
- `customer_count` - number of customers in that country (sorted descending)

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1
df = spark.table("samples.bakehouse.sales_customers")
result_1 = df.groupBy("country").agg(F.count("*").alias("customer_count")).orderBy(F.col("customer_count").desc())
display(result_1)  # replace this

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'country' in cols, "Missing column: country"
assert 'customer_count' in cols, "Missing column: customer_count"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['customer_count'] for r in result_1.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by customer_count descending"
assert all(c > 0 for c in counts), "All customer counts must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Find all **distinct continents** represented in the customer dataset.
Each continent should appear only once in the result.

**Expected output columns:**
- `continent` - unique continent name

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2
result_2 = df.select(F.col('continent')).distinct()

display(result_2)  # replace this

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'continent' in cols, "Missing column: continent"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Too many continents ({cnt}), expected <= 10 distinct values"
continent_vals = [r['continent'] for r in result_2.collect()]
assert len(continent_vals) == len(set(continent_vals)), "continent values must be distinct"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Count customers per **gender**, treating any null gender value as `"Unknown"`.
Use `F.coalesce` or `F.when` to replace nulls before grouping.

**Expected output columns:**
- `gender` - gender label (nulls replaced with "Unknown")
- `count` - number of customers with that gender

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3
result_3 = df.withColumn("gender",F.when(F.col("gender").isNull(),"Unknown").otherwise(F.col("gender"))) \
           .groupBy("gender").agg(F.count("*").alias("count"))
           

display(result_3)  # replace this

In [0]:
result_3 = df.withColumn("gender",F.coalesce(F.col("gender"),F.lit("Unknown"))) \
           .groupBy(F.col("gender")).agg(F.count("*").alias("count"))
           

display(result_3)

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'gender' in cols, "Missing column: gender"
assert 'count' in cols, "Missing column: count"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
gender_vals = [r['gender'] for r in result_3.collect()]
assert None not in gender_vals, "No null gender values allowed - replace with 'Unknown'"
assert all(c > 0 for c in [r['count'] for r in result_3.collect()]), "All counts must be positive"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find customers where the **state field is null or empty string**.
These records may indicate incomplete address information.

**Expected output columns:**
- `customerID` - customer identifier
- `first_name` - customer's first name
- `last_name` - customer's last name
- `country` - customer's country

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4 
result_4 = df.filter(F.col("state").isNotNull()).select("customerID","first_name","last_name","country")

display(result_4) # replace this

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'first_name' in cols, "Missing column: first_name"
assert 'last_name' in cols, "Missing column: last_name"
assert 'country' in cols, "Missing column: country"
cnt = result_4.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find the **top 3 cities** by number of customers.
Include both the city name and the country it belongs to.

**Expected output columns:**
- `city` - city name
- `country` - country the city is in
- `customer_count` - number of customers in that city (top 3 by this value)

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5
result_5 = df.groupBy("city","country").agg(F.count("*").alias("customer_count")).orderBy(F.desc("customer_count")).limit(3)
display(result_5) # replace this

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'city' in cols, "Missing column: city"
assert 'country' in cols, "Missing column: country"
assert 'customer_count' in cols, "Missing column: customer_count"
cnt = result_5.count()
assert cnt == 3, f"Expected exactly 3 rows (top 3), got {cnt}"
counts = [r['customer_count'] for r in result_5.collect()]
assert all(c > 0 for c in counts), "All customer_count values must be positive"
print(f"Problem 5 passed ✓  ({cnt} rows)")